In [ ]:
#!/usr/bin/env python
import os
import shutil
import numpy as np
import pandas as pd

"""
Stable84 pipeline (Verenich-aligned):

For each STABLE log listed in DESC_FILE:
  1) Load raw MuProMAC event log
  2) (Optional) Keep only cases that have a COMPLETE event
  3) Convert to "completion event stream":
        keep rows with status == "running"
        event_time = end_time  (completion timestamp)
  4) Per-case start/end:
        case_start_time    = min(event_time)   [first completion]
        case_complete_time = max(event_time)   [last completion]
  5) Warm-up trim by dropping first WARMUP_FRAC fraction of cases
        ordered by case_start_time
  6) Build prefix rows for ALL remaining cases:
        prefix_index, timesincelastevent, elapsed_time, remaining_time, activity_duration
        open_cases computed on the FULL trimmed set using [case_start_time, case_complete_time]
  7) Split remaining cases by case_start_time into train/val/test
  8) Write:
        <base>_train_prefix.csv
        <base>_val_prefix.csv
        <base>_test_prefix.csv
        <base>_splitmap.csv

This ensures congestion feature `open_cases` reflects the (trimmed) system WIP,
not split-internal WIP.
"""

# -----------------------------
# CONFIG
# -----------------------------
RUN_TAG = "251110"
OUT_ROOT = f"out/{RUN_TAG}"
RESULTS_DIR = os.path.join(OUT_ROOT, "results")

DESC_FILE = os.path.join(RESULTS_DIR, "FIFO_EXP_path_descriptors_machine_and_activity_level_STABLE.csv")

COPY_RAW_LOGS = True
RAW_DEST_DIR = os.path.join(OUT_ROOT, "stable_logs_raw_stable84")

# Output prefixes (train/val/test) for stable84
PREFIX_OUT_DIR = os.path.join(OUT_ROOT, "prefix_datasets_stable84")

# Warm-up trim: drop first X fraction of cases (by case_start_time)
WARMUP_FRAC = 0.17

# Train/Val/Test split fractions (time-ordered cases)
SPLIT = (0.60, 0.20, 0.20)  # must sum to 1.0

# If True, keep only cases that have a COMPLETE event in the raw log
REQUIRE_COMPLETE = True

# Required columns in raw MuProMAC logs
REQUIRED_RAW_COLS = ["case_id", "timestamp", "status", "end_time", "activity", "resource"]

# Status used for activity executions in MuProMAC
RUNNING_STATUS = "running"
COMPLETE_STATUS = "COMPLETE"


# -----------------------------
# Helpers: find logs
# -----------------------------
def normalize_log_name_to_csv_candidates(log_name: str) -> list[str]:
    log_name = str(log_name).strip()
    cands = [log_name]
    if not log_name.lower().endswith(".csv"):
        cands.append(log_name + ".csv")
    return cands

def find_log_file(log_name: str) -> str | None:
    candidates = normalize_log_name_to_csv_candidates(log_name)

    # 1) direct in results dir
    for cand in candidates:
        direct = os.path.join(RESULTS_DIR, cand)
        if os.path.isfile(direct):
            return direct

    # 2) recursive in OUT_ROOT
    cand_set = set(candidates)
    for root, _, files in os.walk(OUT_ROOT):
        for f in files:
            if f in cand_set:
                return os.path.join(root, f)

    return None

def strip_csv_suffix(path_or_name: str) -> str:
    fn = os.path.basename(path_or_name)
    return fn[:-4] if fn.lower().endswith(".csv") else fn


# -----------------------------
# Core logic
# -----------------------------
def keep_complete_cases_raw(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Keep only cases that have at least one COMPLETE row in the RAW log."""
    if not REQUIRE_COMPLETE:
        return df_raw
    complete_cases = set(df_raw.loc[df_raw["status"] == COMPLETE_STATUS, "case_id"].unique())
    return df_raw[df_raw["case_id"].isin(complete_cases)].copy()

def build_completion_stream(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Verenich-style events = activity completions only.
    In MuProMAC: 'running' rows represent activity executions with:
      timestamp = service start
      end_time  = service completion
    We use event_time = end_time as the 'Complete Timestamp'.
    """
    events = df_raw[df_raw["status"] == RUNNING_STATUS].copy()
    if events.empty:
        return events

    events["event_time"] = events["end_time"]

    # Activity duration: completion - start
    events["activity_duration"] = (events["end_time"] - events["timestamp"]).fillna(0.0)

    # sort for stable behavior
    events = events.sort_values(["case_id", "event_time"]).reset_index(drop=True)
    return events

def add_case_start_end(events: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    case_start_time / case_complete_time computed on completion timestamps (event_time),
    matching the Verenich repo's dt_first_last on Complete Timestamp.
    Returns:
      events_with_case_times, per_case_df
    """
    per_case = (
        events.groupby("case_id")["event_time"]
        .agg(["min", "max"])
        .rename(columns={"min": "case_start_time", "max": "case_complete_time"})
        .reset_index()
    )
    events = events.merge(per_case, on="case_id", how="left")
    return events, per_case

def warmup_trim_cases(per_case: pd.DataFrame) -> tuple[list, int, int]:
    """Drop first WARMUP_FRAC of cases ordered by case_start_time."""
    per_case_sorted = per_case.sort_values("case_start_time").reset_index(drop=True)
    case_ids = per_case_sorted["case_id"].to_list()
    n = len(case_ids)
    k_drop = int(np.floor(WARMUP_FRAC * n))
    kept = case_ids[k_drop:]
    return kept, k_drop, n

def compute_open_cases(events: pd.DataFrame, per_case_trimmed: pd.DataFrame) -> pd.DataFrame:
    """
    open_cases(t) = #cases with start_time <= t  -  #cases with end_time <= t
    computed on the FULL trimmed case set, and then merged into events by event_time.
    """
    intervals = per_case_trimmed.rename(columns={"case_start_time": "start_time",
                                                 "case_complete_time": "end_time"}).copy()

    if intervals.empty:
        events["open_cases"] = 0
        return events

    starts = intervals[["start_time"]].sort_values("start_time").reset_index(drop=True)
    starts["n_starts"] = np.arange(1, len(starts) + 1, dtype=np.int64)

    ends = intervals[["end_time"]].sort_values("end_time").reset_index(drop=True)
    ends["n_ends"] = np.arange(1, len(ends) + 1, dtype=np.int64)

    ev = events.sort_values("event_time").reset_index(drop=True)

    ev = pd.merge_asof(ev, starts, left_on="event_time", right_on="start_time", direction="backward")
    ev = pd.merge_asof(ev, ends,   left_on="event_time", right_on="end_time",   direction="backward")

    ev["n_starts"] = ev["n_starts"].fillna(0)
    ev["n_ends"]   = ev["n_ends"].fillna(0)

    ev["open_cases"] = (ev["n_starts"] - ev["n_ends"]).astype(int)

    ev = ev.drop(columns=[c for c in ["start_time", "end_time", "n_starts", "n_ends"] if c in ev.columns])
    return ev

def build_prefix_rows(events: pd.DataFrame) -> pd.DataFrame:
    """Compute per-prefix features (Verenich-style)."""
    events = events.sort_values(["case_id", "event_time"]).reset_index(drop=True)

    events["prefix_index"] = events.groupby("case_id").cumcount() + 1
    events["timesincelastevent"] = events.groupby("case_id")["event_time"].diff().fillna(0.0)

    events["elapsed_time"] = events["event_time"] - events["case_start_time"]
    events["remaining_time"] = events["case_complete_time"] - events["event_time"]

    # Keep output columns consistent
    keep_cols = [
        "case_id",
        "activity",
        "resource",
        "event_time",
        "activity_duration",
        "prefix_index",
        "timesincelastevent",
        "elapsed_time",
        "remaining_time",
        "open_cases",
        "case_start_time",
        "case_complete_time",
    ]

    # keep any meta columns if present
    for extra in ["scenario", "method", "l", "simulation_run", "process"]:
        if extra in events.columns and extra not in keep_cols:
            keep_cols.append(extra)

    out = events[keep_cols].rename(columns={"event_time": "timestamp"})
    return out

def split_cases_time_ordered(per_case_trimmed: pd.DataFrame):
    """
    Split by case_start_time on the TRIMMED set.
    """
    per_case_sorted = per_case_trimmed.sort_values("case_start_time").reset_index(drop=True)
    case_ids = per_case_sorted["case_id"].to_list()

    n2 = len(case_ids)
    n_train = int(np.floor(SPLIT[0] * n2))
    n_val   = int(np.floor(SPLIT[1] * n2))

    train_ids = case_ids[:n_train]
    val_ids   = case_ids[n_train:n_train + n_val]
    test_ids  = case_ids[n_train + n_val:]

    return train_ids, val_ids, test_ids

def write_prefix_splits(prefix_df: pd.DataFrame, base: str, out_dir: str,
                        train_ids, val_ids, test_ids):
    os.makedirs(out_dir, exist_ok=True)

    def sub(ids):
        return prefix_df[prefix_df["case_id"].isin(ids)].copy()

    df_train = sub(train_ids)
    df_val   = sub(val_ids)
    df_test  = sub(test_ids)

    out_train = os.path.join(out_dir, f"{base}_train_prefix.csv")
    out_val   = os.path.join(out_dir, f"{base}_val_prefix.csv")
    out_test  = os.path.join(out_dir, f"{base}_test_prefix.csv")

    df_train.to_csv(out_train, index=False)
    df_val.to_csv(out_val, index=False)
    df_test.to_csv(out_test, index=False)

    splitmap = pd.DataFrame({
        "case_id": train_ids + val_ids + test_ids,
        "split": (["train"] * len(train_ids)) + (["val"] * len(val_ids)) + (["test"] * len(test_ids)),
    })
    out_map = os.path.join(out_dir, f"{base}_splitmap.csv")
    splitmap.to_csv(out_map, index=False)

    return len(df_train), len(df_val), len(df_test), out_train, out_val, out_test, out_map


# -----------------------------
# Main
# -----------------------------
def main():
    if not os.path.isfile(DESC_FILE):
        raise FileNotFoundError(f"Descriptor file not found: {DESC_FILE}")

    df_desc = pd.read_csv(DESC_FILE).replace([np.inf, -np.inf], np.nan)
    if "log_name" not in df_desc.columns:
        raise ValueError("Descriptor file missing required column: log_name")

    log_names = sorted(df_desc["log_name"].dropna().unique().tolist())
    print("Unique logs in DESC:", len(log_names))

    if COPY_RAW_LOGS:
        os.makedirs(RAW_DEST_DIR, exist_ok=True)
    os.makedirs(PREFIX_OUT_DIR, exist_ok=True)

    missing_files = []
    processed = 0
    skipped_no_events = 0

    print(f"\nFound {len(log_names)} STABLE logs listed in:\n  {DESC_FILE}\n")

    for log_name in log_names:
        src = find_log_file(log_name)
        if src is None:
            missing_files.append(log_name)
            continue

        # copy raw using REAL filename
        if COPY_RAW_LOGS:
            dst_raw = os.path.join(RAW_DEST_DIR, os.path.basename(src))
            shutil.copy2(src, dst_raw)
            read_path = dst_raw
        else:
            read_path = src

        base = strip_csv_suffix(read_path)

        df_raw = pd.read_csv(read_path)
        missing_cols = [c for c in REQUIRED_RAW_COLS if c not in df_raw.columns]
        if missing_cols:
            raise ValueError(f"{read_path} missing required columns: {missing_cols}")

        # 1) complete-case filter on RAW log
        df_raw = keep_complete_cases_raw(df_raw)
        if df_raw.empty:
            print(f"[SKIP] {base}: empty after COMPLETE filtering")
            continue

        # 2) completion stream (running rows, with completion time)
        events = build_completion_stream(df_raw)
        if events.empty:
            print(f"[SKIP] {base}: no '{RUNNING_STATUS}' rows -> cannot build prefixes")
            skipped_no_events += 1
            continue

        # 3) case start/end from completion timestamps
        events, per_case = add_case_start_end(events)

        # 4) warm-up trim by case_start_time
        kept_case_ids, k_drop, n_total_cases = warmup_trim_cases(per_case)
        per_case_trimmed = per_case[per_case["case_id"].isin(kept_case_ids)].copy()
        events_trimmed = events[events["case_id"].isin(kept_case_ids)].copy()

        if per_case_trimmed.empty or events_trimmed.empty:
            print(f"[SKIP] {base}: empty after warm-up trimming")
            continue

        # 5) open_cases computed on FULL trimmed set
        events_trimmed = compute_open_cases(events_trimmed, per_case_trimmed)

        # 6) build prefix rows
        prefix_df = build_prefix_rows(events_trimmed)

        # 7) split cases on trimmed case_start_time
        train_ids, val_ids, test_ids = split_cases_time_ordered(per_case_trimmed)

        # guard: empty splits
        if len(train_ids) == 0 or len(val_ids) == 0 or len(test_ids) == 0:
            print(f"[WARN] {base}: split produced empty subset(s). "
                  f"(cases total={n_total_cases}, warmup_dropped={k_drop}, remaining={len(per_case_trimmed)})")

        ntr, nv, nts, out_train, out_val, out_test, out_map = write_prefix_splits(
            prefix_df, base, PREFIX_OUT_DIR, train_ids, val_ids, test_ids
        )

        processed += 1
        print(f"[OK] {base}")
        print(f"     cases total={n_total_cases}, warmup_dropped={k_drop}, remaining={len(per_case_trimmed)}")
        print(f"     train/val/test cases={len(train_ids)}/{len(val_ids)}/{len(test_ids)}")
        print(f"     prefix rows train/val/test={ntr}/{nv}/{nts}")
        # print(f"     -> {out_train}")
        # print(f"     -> {out_val}")
        # print(f"     -> {out_test}")

    print("\n" + "=" * 70)
    print(f"Done. Processed {processed}/{len(log_names)} logs.")
    print(f"Prefix outputs in: {PREFIX_OUT_DIR}")
    if COPY_RAW_LOGS:
        print(f"Copied raw logs in: {RAW_DEST_DIR}")
    if skipped_no_events:
        print(f"Skipped (no running rows): {skipped_no_events}")

    if missing_files:
        print("\nWARNING: could not find these log files (first 50):")
        for m in missing_files[:50]:
            print("  -", m)


if __name__ == "__main__":
    main()


Unique logs in DESC: 84

Found 84 STABLE logs listed in:
  out/251110\results\FIFO_EXP_path_descriptors_machine_and_activity_level_STABLE.csv

[OK] FIFO_EXP_l0.28_dedicated_C1_V18_A5_QC97_identical
     cases total=8424, warmup_dropped=1432, train/val/test cases=4195/1398/1399
     events train/val/test=89757/30030/30054
[OK] FIFO_EXP_l0.28_dedicated_C1_V18_A5_QC97_mild_all
     cases total=8367, warmup_dropped=1422, train/val/test cases=4167/1389/1389
     events train/val/test=89307/29856/29871
[OK] FIFO_EXP_l0.28_dedicated_C1_V18_A8_QC97_identical
     cases total=8368, warmup_dropped=1422, train/val/test cases=4167/1389/1390
     events train/val/test=132102/44094/44134
[OK] FIFO_EXP_l0.28_dedicated_C1_V18_A8_QC97_mild_all
     cases total=8423, warmup_dropped=1431, train/val/test cases=4195/1398/1399
     events train/val/test=133030/44361/44422
[OK] FIFO_EXP_l0.28_dedicated_C1_V6_A12_QC100_identical
     cases total=8316, warmup_dropped=1413, train/val/test cases=4141/1380/1382
 